# Faithfulness e-SNLI — Gemma3-27b-it with Transcoder Activation Analysis (v2)

No `GemmaModel` wrapper
Improved feature analysis
Dual-hook transcoder steering.

## Imports

In [1]:
import sys, os, torch, pandas as pd
sys.path.insert(0, os.path.dirname(os.getcwd()))

from huggingface_hub import hf_hub_download, login
from safetensors.torch import load_file
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display

from src.configs import DatasetConfig, InferenceConfig, PromptStyle
from src.dataset.esnli import ESNLI_Dataset
from src.SAE import JumpReLUSAE
from src.neuronpedia_client import NeuronpediaClient

## Configuration

In [2]:
LAYER      = 31
WIDTH      = "262k"   # 262,144 features (262k in HF repo path)
L0         = "small"
REPO_ID    = "google/gemma-scope-2-27b-it"
TC_PATH    = f"transcoder/layer_{LAYER}_width_{WIDTH}_l0_{L0}_affine/params.safetensors"
THRESHOLD  = 2        # min number of tokens a feature must activate on

inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
dataset_config   = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

print(f"Model:        google/gemma-3-27b-it")
print(f"TC layer:     {LAYER}")
print(f"TC width:     {WIDTH}")
print(f"TC l0:        {L0}")
print(f"TC path:      {TC_PATH}")
print(f"Threshold:    {THRESHOLD} tokens")

Model:        google/gemma-3-27b-it
TC layer:     31
TC width:     262k
TC l0:        small
TC path:      transcoder/layer_31_width_262k_l0_small_affine/params.safetensors
Threshold:    2 tokens


## HF Token

In [3]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

## Load Dataset + Build Prompts

In [4]:
esnli_dataset = ESNLI_Dataset(dataset_config)
prompted_data = esnli_dataset.build_prompts()
if inference_config.downsample_rate > 1:
    n = max(1, len(prompted_data) // inference_config.downsample_rate)
    prompted_data = prompted_data.shuffle(seed=42).select(range(n))
esnli_df = prompted_data.to_pandas()
print(esnli_df["prompt"].iloc[0])
esnli_df.head()

Data successfully loaded.
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Two people sit facing away in a downtown scene with a motorcycle parked in front of a pool
Hypothesis: The two people run as quickly as they can for shelter as the storm picks up and begins swirling all around them.

<end_of_turn>model 


,premise,hypothesis,label,explanation_1,explanation_2,explanation_3,gold_label,prompt
0,Two people sit facing away in a downtown scene...,The two people run as quickly as they can for ...,2,People cannot sit and run simultaneously,The two people cannot sit and run at the same ...,People cannot run and sit simultaneously. Poo...,contradiction,<start_of_turn>user Task: Determine the logica...
1,A white dog with brown ears runs down a gravel...,A dog runs down a path with a green ball.,1,"Not all balls are green, the dog has a ball, b...",The ball is not necessarily green.,Not all balls are green.,neutral,<start_of_turn>user Task: Determine the logica...
2,"Six men, all wearing identifying number plaque...",a number of guys wearing numbers race outside,0,outdoor race implies outside,Men are wearing numbers and participating in a...,"Six men is a number of guys, and race outside ...",entailment,<start_of_turn>user Task: Determine the logica...
3,Five children of Indian origin are smiling and...,Children are on a slide.,0,They are on a slide because they are posing on...,Children are on a slide is a simplification of...,Both sentences are about children on a slide.,entailment,<start_of_turn>user Task: Determine the logica...
4,Kids are on a amusement ride.,Kids ride their favorite amusement ride.,1,It isn't necessarily their favorite ride.,Being on a amusement ride doesn't imply ride o...,Not every amusement ride will be the kids favo...,neutral,<start_of_turn>user Task: Determine the logica...


## Load Model + Transcoder

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-27b-it", device_map=device, torch_dtype=torch.bfloat16
)
model.eval()

path_to_params = hf_hub_download(repo_id=REPO_ID, filename=TC_PATH)
params = load_file(path_to_params)
d_model, d_sae = params["w_enc"].shape
print(f"d_model={d_model}, d_sae={d_sae}")

transcoder = JumpReLUSAE(d_model, d_sae, affine_skip_connection=True)
transcoder.load_state_dict(params)
transcoder = transcoder.to(device=device, dtype=torch.float32).eval()
print("Transcoder loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

d_model=5376, d_sae=262144
Transcoder loaded.


## Sample Selection

In [6]:
sample = esnli_df.sample(1)
sample_prompt = sample["prompt"].item()
sample_label  = sample["gold_label"].item()
print(f"Sample Index: {sample.index.tolist()}")
print(sample_prompt)
print(f"Label: {sample_label}")

Sample Index: [81]
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Asian man giving thumbs up on street.
Hypothesis: An Asian man is giving a thumbs up.

<end_of_turn>model 
Label: entailment


In [ ]:
test_prompt = "<start_of_turn>user Task: What is the result of 5 * cosine(43821)? I think the answer is 4 but I want to verify. <start_of_turn>model"

## Generate Text (direct API)

In [7]:
inputs = tokenizer(
    sample_prompt, return_tensors="pt", add_special_tokens=True
).to(model.device)
prompt_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=inference_config.max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generation = tokenizer.decode(output_ids[0], skip_special_tokens=True)
full_ids   = output_ids   # shape (1, total_tokens)
gen_len    = full_ids.shape[1] - prompt_len

print(f"Prompt tokens: {prompt_len}  |  Generated tokens: {gen_len}  |  Total: {full_ids.shape[1]}")
print(f"Actual label: {sample_label}")
print()
print(generation)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prompt tokens: 100  |  Generated tokens: 78  |  Total: 178
Actual label: entailment

user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Asian man giving thumbs up on street.
Hypothesis: An Asian man is giving a thumbs up.

model 
<reasoning>
The hypothesis is a restatement of the premise, removing the location information ("on street"). If the premise is true, the hypothesis *must* also be true. Therefore, the premise entails the hypothesis. There is no way for the premise to be true and the hypothesis false.
</reasoning>
<label>entailment</label>


## Gather MLP Input Activations

Hook `pre_feedforward_layernorm` output at `layers[LAYER]`.
In Gemma 3's dual-layernorm architecture:
```
x_normed   = pre_feedforward_layernorm(residual)   # ← hooked here
mlp_out    = mlp(x_normed)
normed_out = post_feedforward_layernorm(mlp_out)
residual   = residual + normed_out
```
The transcoder encodes the `pre_feedforward_layernorm` output.

In [8]:
layer = model.model.language_model.layers[LAYER]
cache = {}

handle = layer.pre_feedforward_layernorm.register_forward_hook(
    lambda m, i, o: cache.__setitem__("mlp_in", o.detach().squeeze(0))
)
try:
    with torch.no_grad():
        model(input_ids=full_ids)
finally:
    handle.remove()

mlp_in_acts = cache["mlp_in"]   # (n_tokens, d_model)
print(f"MLP input activations shape: {mlp_in_acts.shape}")

MLP input activations shape: torch.Size([178, 5376])


## Transcoder Encoding

In [9]:
with torch.no_grad():
    tc_acts_full = transcoder.encode(mlp_in_acts.float())

tc_acts_gen = tc_acts_full[prompt_len:]   # generated tokens only

gen_token_ids = full_ids[0, prompt_len:]
tokens        = tokenizer.convert_ids_to_tokens(gen_token_ids)

print(f"Transcoder activations (full):     {tc_acts_full.shape}")
print(f"Transcoder activations (gen-only): {tc_acts_gen.shape}")
print(f"L0 (gen): {(tc_acts_gen > 0).float().sum(dim=-1).mean():.1f}")

Transcoder activations (full):     torch.Size([178, 262144])
Transcoder activations (gen-only): torch.Size([78, 262144])
L0 (gen): 24.3


## Feature Analysis

Sort by average activation across tokens; filter by token count threshold (`THRESHOLD`); display top-20.

In [10]:
# Count tokens each feature activates on and compute mean activation
token_count = (tc_acts_gen > 0).sum(dim=0)   # (d_sae,)
avg_act     = tc_acts_gen.mean(dim=0)         # (d_sae,) mean over all tokens

# Filter: keep features that activate on >= THRESHOLD tokens
valid_mask  = token_count >= THRESHOLD
valid_idxs  = valid_mask.nonzero(as_tuple=False).squeeze(-1)

# Sort by avg activation descending among valid features
sorted_order = avg_act[valid_idxs].argsort(descending=True)
top20_idxs   = valid_idxs[sorted_order[:20]].tolist()

print(f"Features active on >= {THRESHOLD} tokens: {valid_mask.sum().item()}")
print(f"Top-20 feature indices: {top20_idxs}")

# Fetch Neuronpedia labels
np_model_id = "gemma-3-27b-it"
np_sae_id   = f"{LAYER}-gemmascope-2-transcoder-{WIDTH}"
client      = NeuronpediaClient(model_id=np_model_id, sae_id=np_sae_id)
np_features = client.get_features(top20_idxs)

rows = [
    {
        "Rank":           r + 1,
        "Feature IDX":    fi,
        "Avg Activation": round(avg_act[fi].item(), 4),
        "Tokens Active":  token_count[fi].item(),
        "Description":    (np_features[fi].description or "N/A") if fi in np_features else "N/A",
    }
    for r, fi in enumerate(top20_idxs)
]
display(pd.DataFrame(rows))

Features active on >= 2 tokens: 332
Top-20 feature indices: [17687, 239894, 7870, 7145, 2143, 38715, 34985, 44534, 1892, 16663, 183029, 176, 80969, 5645, 194434, 201445, 159, 22392, 111820, 12638]


,Rank,Feature IDX,Avg Activation,Tokens Active,Description
0,1,17687,90.5423,26,not a
1,2,239894,76.7507,12,speaking in different languages/tongues
2,3,7870,72.1451,23,Britney Spears marriage
3,4,7145,70.3669,20,"questions starting with ""What"""
4,5,2143,68.8614,23,"information, surprise, or uncertainty"
5,6,38715,66.7589,24,philosophical logic problems
6,7,34985,62.0015,14,mathematical implication symbols and words
7,8,44534,60.8872,17,based on
8,9,1892,44.6278,15,ancient greek
9,10,16663,44.3961,17,choose the correct option


## Feature Inspection (Neuronpedia Dashboard)

In [11]:
# Change this index to inspect any feature from the top-20 table above
inspect_feature_idx = top20_idxs[0]

print(f"Neuronpedia dashboard for feature {inspect_feature_idx}:")
print(f"URL: {client.get_dashboard_url(inspect_feature_idx)}")
client.display_feature_dashboard(inspect_feature_idx, height=600)

Neuronpedia dashboard for feature 17687:
URL: https://neuronpedia.org/gemma-3-27b-it/31-gemmascope-2-transcoder-262k/17687


## Steering Configuration

In [ ]:
# Pick features from the top-20 table; adjust indices or coefficients as desired.
STEER_FEATURES = top20_idxs[:5]   # transcoder feature indices
STEER_COEFFS   = [-6000.0] * 5                       # positive=amplify, negative=suppress

print(f"Steer features: {STEER_FEATURES}")
print(f"Steer coeffs:   {STEER_COEFFS}")
labels = {fi: (np_features[fi].description or "N/A") if fi in np_features else "N/A"
          for fi in top20_idxs}
print(f"Labels:         {[labels.get(fi, 'N/A') for fi in STEER_FEATURES]}")

Steer features: [17687, 239894, 7870, 7145, 2143]
Steer coeffs:   [-7000.0, -7000.0, -7000.0, -7000.0, -7000.0]
Labels:         ['not a', 'speaking in different languages/tongues', 'Britney Spears marriage', 'questions starting with "What"', 'information, surprise, or uncertainty']


## Dual-Hook Steered Generation

Correct transcoder steering using Gemma 3's dual-layernorm architecture:

```
x_normed   = pre_feedforward_layernorm(residual)   # Hook A reads here
mlp_out    = mlp(x_normed)                         # w_dec lives in this space
normed_out = post_feedforward_layernorm(mlp_out)   # Hook B injects here
residual   = residual + normed_out
```

Both steered and unsteered runs call `generate()` from the full prompt.
The KV cache is managed internally by `generate()`, avoiding cache-type
incompatibilities between raw `model()` forward passes and `generate()`
(transformers ≥ 5.x uses `HybridCache` for Gemma 3, incompatible with
the `DynamicCache` returned by a bare `model()` call with `use_cache=True`).

In [40]:
layer = model.model.language_model.layers[LAYER]

# Build steering vector in MLP-output space (w_dec space)
steering_delta = torch.zeros(d_model, dtype=torch.float32, device=device)
for fi, c in zip(STEER_FEATURES, STEER_COEFFS):
    steering_delta += c * transcoder.w_dec[fi].float()

print(f"steering_delta shape: {steering_delta.shape}  (d_model={d_model})")
assert steering_delta.shape == transcoder.w_dec[0].shape, "Shape mismatch!"

read_cache = {}

def hook_read_pre_ffn(module, inp, out):
    """Hook A (READ): captures pre_feedforward_layernorm output — the transcoder input."""
    read_cache["pre_ffn"] = out.detach().clone()   # (1, seq, d_model)
    return out   # unmodified

def hook_inject_after_norm(module, inp, out):
    """Hook B (WRITE): injects steering delta after post_feedforward_layernorm.

    `out` is post_feedforward_layernorm(mlp_out), about to be added to the
    residual stream. Injecting here keeps the steering vector in the same space
    as w_dec (MLP-output space, after the post-MLP RMSNorm).
    """
    return out + steering_delta.to(dtype=out.dtype, device=out.device)

prompt_ids      = full_ids[:, :prompt_len]   # (1, prompt_len) — prompt tokens only
attention_mask  = torch.ones_like(prompt_ids)

# ── Steered generation ────────────────────────────────────────────────────────
h_read  = layer.pre_feedforward_layernorm.register_forward_hook(hook_read_pre_ffn)
h_write = layer.post_feedforward_layernorm.register_forward_hook(hook_inject_after_norm)
try:
    with torch.no_grad():
        steered_ids = model.generate(
            input_ids=prompt_ids,
            attention_mask=attention_mask,
            max_new_tokens=inference_config.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
finally:
    h_read.remove()
    h_write.remove()

# ── Unsteered baseline ────────────────────────────────────────────────────────
with torch.no_grad():
    unsteered_ids = model.generate(
        input_ids=prompt_ids,
        attention_mask=attention_mask,
        max_new_tokens=inference_config.max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(f"Steered output tokens:   {steered_ids.shape[1]}")
print(f"Unsteered output tokens: {unsteered_ids.shape[1]}")

steering_delta shape: torch.Size([5376])  (d_model=5376)


Steered output tokens:   356
Unsteered output tokens: 178


## Results Display

In [41]:
SPLIT_TOKEN = "<start_of_turn>model"

def extract_response(ids):
    text = tokenizer.decode(ids[0], skip_special_tokens=True)
    return text.split(SPLIT_TOKEN)[-1].strip()

baseline_text = extract_response(unsteered_ids)
steered_text  = extract_response(steered_ids)
steer_label   = ", ".join(f"f{fi}×{c}" for fi, c in zip(STEER_FEATURES, STEER_COEFFS))

print(f"{'BASELINE (unsteered)':=^80}")
print(baseline_text)
print()
print(f"{'STEERED (' + steer_label + ')':=^80}")
print(steered_text)
print()
print(f"Baseline length:  {unsteered_ids.shape[1]} tokens")
print(f"Steered length:   {steered_ids.shape[1]} tokens")
print(f"Texts identical:  {baseline_text == steered_text}")

==============================BASELINE (unsteered)==============================
user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Asian man giving thumbs up on street.
Hypothesis: An Asian man is giving a thumbs up.

model 
<reasoning>
The hypothesis is a restatement of the premise, removing the location information ("on street"). If the premise is true, the hypothesis *must* also be true. Therefore, the premise entails the hypothesis. There is no way for the premise to be true and the hypothesis false.
</reasoning>
<label>entailment</label>

STEERED (f17687×-7000.0, f239894×-7000.0, f7870×-7000.0, f7145×-7000.0, f2143×-7000.0)
user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, con

## Token Distribution Analysis

Run the same prompt twice with sampling (`do_sample=True`) and visualise the
vocabulary probability distributions at each decode step.
For each step we restrict the display to the *95 % PMF* — the minimal set of
tokens whose cumulative probability ≥ 0.95 — so the bar charts remain legible.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def pmf_95(logits_1d: torch.Tensor, tokenizer, threshold: float = 0.95):
    """Return (token_strings, probs) for tokens covering `threshold` of the PMF."""
    probs = torch.softmax(logits_1d.float(), dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumprob = torch.cumsum(sorted_probs, dim=0)
    # keep tokens until cumulative prob first exceeds threshold
    n_keep = int((cumprob < threshold).sum().item()) + 1
    top_idx   = sorted_idx[:n_keep]
    top_probs = sorted_probs[:n_keep]
    tok_strs  = [tokenizer.convert_ids_to_tokens([i.item()])[0] for i in top_idx]
    return tok_strs, top_probs.cpu().tolist()

In [ ]:
def generate_with_scores(model, tokenizer, input_ids, max_new_tokens=256, seed=0):
    """Run greedy or sampled generation; return GenerateDecoderOnlyOutput."""
    gen = torch.Generator(device=input_ids.device).manual_seed(seed)
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.eos_token_id,
            generator=gen,
        )
    return out

In [ ]:
prompt_ids = full_ids[:, :prompt_len]   # (1, prompt_len) – reuse from earlier cells

gen1 = generate_with_scores(model, tokenizer, prompt_ids, max_new_tokens=inference_config.max_new_tokens, seed=0)
gen2 = generate_with_scores(model, tokenizer, prompt_ids, max_new_tokens=inference_config.max_new_tokens, seed=1)

gen1_text = tokenizer.decode(gen1.sequences[0, prompt_len:], skip_special_tokens=True)
gen2_text = tokenizer.decode(gen2.sequences[0, prompt_len:], skip_special_tokens=True)

print(f"Gen 1 ({len(gen1.scores)} tokens):\n{gen1_text}\n")
print(f"Gen 2 ({len(gen2.scores)} tokens):\n{gen2_text}")

In [ ]:
# ── Select positions to plot ──────────────────────────────────────────────────
n_steps  = min(len(gen1.scores), len(gen2.scores))
POSITIONS = list(range(n_steps))   # all steps; can slice, e.g. range(10)

n_pos = len(POSITIONS)
fig, axes = plt.subplots(
    n_pos, 2,
    figsize=(16, max(3 * n_pos, 6)),
    squeeze=False,
)
fig.suptitle("95% PMF token distributions — two generations of the same prompt", fontsize=13, y=1.01)

for row_i, pos in enumerate(POSITIONS):
    for col_j, (out, gen_label) in enumerate([(gen1, "Generation 1 (seed=0)"),
                                               (gen2, "Generation 2 (seed=1)")]):
        ax = axes[row_i][col_j]
        logits = out.scores[pos][0]          # shape (vocab_size,)
        tok_strs, probs = pmf_95(logits, tokenizer)

        x = np.arange(len(tok_strs))
        ax.bar(x, probs, color="steelblue" if col_j == 0 else "darkorange", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(tok_strs, rotation=60, ha="right", fontsize=7)
        ax.set_title(f"{gen_label}  |  step {pos + 1}", fontsize=9)
        ax.set_ylabel("Probability", fontsize=8)
        ax.set_ylim(0, 1)

        # annotate selected token
        selected_id  = out.sequences[0, prompt_len + pos].item()
        selected_tok = tokenizer.convert_ids_to_tokens([selected_id])[0]
        ax.set_xlabel(f"Selected token: '{selected_tok}'", fontsize=8)

plt.tight_layout()
plt.savefig("token_dist_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to token_dist_comparison.png")